In [1]:
import pandas as pd

In [2]:
def media_e_desvio_padrao(df):
    colunas_numericas = df.select_dtypes(include="number")
    return pd.DataFrame({
        "media": colunas_numericas.mean().round(2),
        "desvio_padrao": colunas_numericas.std().round(2)
    })

In [3]:
baseline_scd = pd.read_csv("resultados-exp-01-base-dblp.csv")
baseline_ncvr = pd.read_csv("resultados-exp-01-base-ncvoter.csv")
baseline_music = pd.read_csv("resultados-exp-01-base-musicbrainz.csv")
media_desvio_scd = media_e_desvio_padrao(baseline_scd)
media_desvio_ncvr = media_e_desvio_padrao(baseline_ncvr)
media_desvio_music = media_e_desvio_padrao(baseline_music)

In [4]:
descarte_tempo_scd = pd.read_csv("resultados-exp-02-descarte-tempo-dblp.csv")
descarte_tempo_ncvr = pd.read_csv("resultados-exp-02-descarte-tempo-ncvoter.csv")
descarte_tempo_music = pd.read_csv("resultados-exp-02-descarte-tempo-musicbrainz.csv")
media_desvio_tempo_scd = media_e_desvio_padrao(descarte_tempo_scd)
media_desvio_tempo_ncvr = media_e_desvio_padrao(descarte_tempo_ncvr)
media_desvio_tempo_music = media_e_desvio_padrao(descarte_tempo_music)

In [5]:
descarte_bloco_scd = pd.read_csv("resultados-exp-03-descarte-bloco-dblp.csv")
descarte_bloco_ncvr = pd.read_csv("resultados-exp-03-descarte-bloco-ncvoter.csv")
descarte_bloco_music = pd.read_csv("resultados-exp-03-descarte-bloco-musicbrainz.csv")

media_desvio_bloco_scd = media_e_desvio_padrao(descarte_bloco_scd)
media_desvio_bloco_ncvr = media_e_desvio_padrao(descarte_bloco_ncvr)
media_desvio_bloco_music = media_e_desvio_padrao(descarte_bloco_music)

In [ ]:
experimentos = {
    "Base": {
        "DBLP": baseline_scd,
        "NCVoter": baseline_ncvr,
        "MusicBrainz": baseline_music,
    },
    "Descarte por tempo": {
        "DBLP": descarte_tempo_scd,
        "NCVoter": descarte_tempo_ncvr,
        "MusicBrainz": descarte_tempo_music,
    },
    "Descarte por bloco": {
        "DBLP": descarte_bloco_scd,
        "NCVoter": descarte_bloco_ncvr,
        "MusicBrainz": descarte_bloco_music,
    },
}

metricas = ["blocking_s", "matching_s", "pairs_no", "tp", "recall", "precision"]
nomes_metricas = {
    "blocking_s": "Block (s)",
    "matching_s": "Match (s)",
    "pairs_no": "#Pairs",
    "tp": "TP",
    "recall": "Recall",
    "precision": "Precision",
}
colunas_tabela = ["Dataset", "Technique", "Block (s)", "Match (s)", "#Pairs", "TP", "Recall", "Precision"]

linhas_tabela = []
for tecnica, datasets in experimentos.items():
    for dataset, resultados in datasets.items():
        linha = {"Dataset": dataset, "Technique": tecnica}
        for metrica in metricas:
            media = resultados[metrica].mean()
            desvio = resultados[metrica].std()
            casas = 3 if metrica in ("recall", "precision") else 2
            linha[nomes_metricas[metrica]] = f"{media:.{casas}f} +/- {desvio:.{casas}f}"
        linhas_tabela.append(linha)

tabela_comparativa = pd.DataFrame(linhas_tabela, columns=colunas_tabela)
tabela_comparativa 


In [7]:
#tabela_comparativa.to_csv("tabela_comparativa.csv", index=False)

In [32]:
from scipy.stats import mannwhitneyu
import pandas as pd

metrics = {
    "blocking_s": "Blocking",
    "matching_s": "Matching",
    "recall": "Recall",
    "precision": "Precision",
    "pairs_no": "Candidate pairs"
}

def tabela_comparacao(base_method, comparison_method):
    rows = []

    for dataset in ["DBLP", "NCVoter", "MusicBrainz"]:
        base = experimentos[base_method][dataset]
        comp = experimentos[comparison_method][dataset]

        row = {"Dataset": dataset}

        for metric, label in metrics.items():
            if base[metric].nunique() == 1 and comp[metric].nunique() == 1:
                p = 1.0
            else:
                _, p = mannwhitneyu(
                    base[metric],
                    comp[metric],
                    alternative="two-sided"
                )

            row[label] = p

        rows.append(row)

    return pd.DataFrame(rows).set_index("Dataset")


# Baseline vs Entity Eviction
entity_eviction = tabela_comparacao(
    "Base",
    "Descarte por tempo"
)

# Baseline vs Block Eviction
block_eviction = tabela_comparacao(
    "Base",
    "Descarte por bloco"
)

print("Baseline vs Entity Eviction")
display(entity_eviction.round(4))

print("Baseline vs Block Eviction")
display(block_eviction.round(4))

Baseline vs Entity Eviction


,Blocking,Matching,Recall,Precision,Candidate pairs
Dataset,,,,,
DBLP,0.0028,0.0002,0.0151,0.0757,0.0757
NCVoter,0.0002,0.0002,1.0000,0.5708,0.5708
MusicBrainz,0.0002,0.0002,1.0000,0.1859,0.1859


Baseline vs Block Eviction


,Blocking,Matching,Recall,Precision,Candidate pairs
Dataset,,,,,
DBLP,0.0028,0.0002,0.0002,0.0002,0.0002
NCVoter,0.0002,0.0002,1.0000,0.0002,0.0002
MusicBrainz,0.0539,0.0002,1.0000,0.0002,0.0002


In [33]:
descarte_total_scd = pd.read_csv("resultados-exp-descarte-velho-dblp.csv")
descarte_total_ncvr = pd.read_csv("resultados-exp-descarte-velho-ncvoter.csv")
descarte_total_music = pd.read_csv("resultados-exp-descarte-velho-musicbrainz.csv")

media_desvio_total_scd = media_e_desvio_padrao(descarte_total_scd)
media_desvio_total_ncvr = media_e_desvio_padrao(descarte_total_ncvr)
media_desvio_total_music = media_e_desvio_padrao(descarte_total_music)

In [34]:
media_desvio_total_scd

,media,desvio_padrao
blocking_s,6.31,0.14
matching_s,169.67,3.60
tp,5286.50,6.95
fp,839.50,41.63
pairs_no,24625.60,175.42
recall,0.99,0.00
precision,0.86,0.01
replica,5.50,3.03
seed,20260911.50,3.03


In [35]:
experimentos2 = {
    "Base": {
        "DBLP": baseline_scd,
        "NCVoter": baseline_ncvr,
        "MusicBrainz": baseline_music,
    },
    "Descarte Global": {
        "DBLP": descarte_total_scd,
        "NCVoter": descarte_total_ncvr,
        "MusicBrainz": descarte_total_music,
    },
}

metricas = ["blocking_s", "matching_s", "pairs_no", "tp", "recall", "precision"]
nomes_metricas = {
    "blocking_s": "Block (s)",
    "matching_s": "Match (s)",
    "pairs_no": "#Pairs",
    "tp": "TP",
    "recall": "Recall",
    "precision": "Precision",
}
colunas_tabela = ["Dataset", "Technique", "Block (s)", "Match (s)", "#Pairs", "TP", "Recall", "Precision"]

linhas_tabela = []
for tecnica, datasets in experimentos2.items():
    for dataset, resultados in datasets.items():
        linha = {"Dataset": dataset, "Technique": tecnica}
        for metrica in metricas:
            media = resultados[metrica].mean()
            desvio = resultados[metrica].std()
            casas = 3 if metrica in ("recall", "precision") else 2
            linha[nomes_metricas[metrica]] = f"{media:.{casas}f} +/- {desvio:.{casas}f}"
        linhas_tabela.append(linha)

tabela_comparativa = pd.DataFrame(linhas_tabela, columns=colunas_tabela)
tabela_comparativa 


,Dataset,Technique,Block (s),Match (s),#Pairs,TP,Recall,Precision
0,DBLP,Base,5.74 +/- 0.14,155.86 +/- 1.42,75527.10 +/- 1044.15,4978.20 +/- 4.73,0.931 +/- 0.001,0.066 +/- 0.001
1,NCVoter,Base,31.06 +/- 0.11,127.76 +/- 1.05,168228.10 +/- 1120.29,38662.00 +/- 0.00,0.945 +/- 0.000,0.230 +/- 0.002
2,MusicBrainz,Base,360.87 +/- 0.69,809.55 +/- 2.07,42744.20 +/- 137.63,20578.00 +/- 0.00,0.106 +/- 0.000,0.481 +/- 0.002
3,DBLP,Descarte Global,6.31 +/- 0.14,169.67 +/- 3.60,24625.60 +/- 175.42,5286.50 +/- 6.95,0.989 +/- 0.001,0.863 +/- 0.006
4,NCVoter,Descarte Global,52.41 +/- 1.82,80.36 +/- 2.30,128081.40 +/- 837.93,40892.00 +/- 0.00,1.000 +/- 0.000,0.483 +/- 0.003
5,MusicBrainz,Descarte Global,404.30 +/- 6.67,541.23 +/- 6.43,202118.10 +/- 80.74,193464.20 +/- 1.40,1.000 +/- 0.000,0.978 +/- 0.000
